# Thiếu văn minh: Qwen2.5 7B Instruct

So sánh zero-shot, few-shot (4 ví dụ mỗi lớp từ UIT-ViCTSD) và few-shot có hướng dẫn suy luận.
Ba prompt dùng cùng một định nghĩa thiếu văn minh (viết trong prompt) và đưa bình luận cha vào làm ngữ cảnh.
Điểm là xác suất của token `1` sau khi chuẩn hóa trên đúng hai token `0` và `1`.

Gắn private dataset từ `python -m src.classifiers.incivility kaggle-bundle` sau khi chốt nhãn dev.
Bật Internet và GPU T4. Mặc định notebook chấm cả ba cấu hình trên dev.

Nếu chọn Qwen làm model cuối, đặt `VISIM_INCIVILITY_PHASE=corpus` và
`VISIM_INCIVILITY_QWEN_MODE` thành `zero`, `few` hoặc `cot` trước khi chạy.
Biến `VISIM_INCIVILITY_QWEN_BATCH` cho phép đổi batch (mặc định 8).
Chép các file scores và JSON từ `/kaggle/working` vào `results/incivility/kaggle/`.
Không dùng đầu ra của notebook làm quyết định sửa nhãn dev.

In [ ]:
%pip install -q transformers==4.57.1 accelerate==1.11.0 bitsandbytes==0.48.2

In [ ]:
import hashlib
import json
import os
import platform
import time
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch

PHASE = os.environ.get("VISIM_INCIVILITY_PHASE", "dev")
if PHASE not in ("dev", "corpus"):
    raise ValueError("VISIM_INCIVILITY_PHASE must be dev or corpus")

kaggle_input = Path("/kaggle/input")
if "VISIM_INCIVILITY_DATA" in os.environ:
    data_dir = Path(os.environ["VISIM_INCIVILITY_DATA"])
elif kaggle_input.is_dir():
    manifests = sorted(kaggle_input.rglob("incivility_manifest.json"))
    if len(manifests) != 1:
        raise ValueError("Attach exactly one VISIM incivility dataset")
    data_dir = manifests[0].parent
else:
    data_dir = Path("data/kaggle/incivility")

out_dir = Path(os.environ.get(
    "VISIM_INCIVILITY_OUTPUT",
    "/kaggle/working" if kaggle_input.is_dir() else "results/incivility/kaggle",
))
out_dir.mkdir(parents=True, exist_ok=True)


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


manifest = json.loads((data_dir / "incivility_manifest.json").read_text(encoding="utf-8"))


def read_data(name):
    path = data_dir / f"incivility_{name}.csv"
    hash_key = "bundle_train_sha256" if name == "train" else f"{name}_sha256"
    if sha256(path) != manifest[hash_key]:
        raise ValueError(f"{path.name} does not match the input manifest")
    frame = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    if frame["id"].duplicated().any():
        raise ValueError(f"{path.name} contains duplicate IDs")
    return frame


def write_scores(frame, score, config):
    if len(score) != len(frame) or not (np.isfinite(score).all() and ((0 <= score) & (score <= 1)).all()):
        raise ValueError(f"invalid scores for {config}")
    path = out_dir / f"{PHASE}_scores_{config}.csv"
    temporary = path.with_suffix(".csv.tmp")
    pd.DataFrame({
        "id": frame["id"], "pred": (score >= 0.5).astype(int), "score": np.round(score, 4),
    }).to_csv(temporary, index=False, encoding="utf-8-sig")
    temporary.replace(path)
    return path


def write_run(family, configs, files, **settings):
    report = {
        "phase": PHASE, "data": manifest, "configs": configs,
        "output_sha256": {p.name: sha256(p) for p in files},
        "where": "Kaggle" if kaggle_input.is_dir() else "local GPU",
        "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda,
        "python": platform.python_version(), "torch": str(torch.__version__),
        "transformers": version("transformers"), **settings,
    }
    path = out_dir / f"{family}_{PHASE}.json"
    temporary = path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)
    print(path)


if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU before running this notebook.")
device = torch.device("cuda:0")
print("Data:", data_dir, "| Output:", out_dir, "| GPU:", torch.cuda.get_device_name(device))

In [ ]:
import re
import unicodedata

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-7B-Instruct"
REVISION = "a09a35458c702b33eeacc393d103063234e8bc28"
SEED = 42
MODES = ("zero", "few", "cot")
BATCH = int(os.environ.get("VISIM_INCIVILITY_QWEN_BATCH", "8"))
MAX_COMMENT, MAX_PARENT, MAX_EXAMPLE, MAX_REASON = 192, 128, 96, 64
LABEL_PREFIX = "Nhãn: "
CORPUS_MODE = os.environ.get("VISIM_INCIVILITY_QWEN_MODE", "zero")
if BATCH < 1 or CORPUS_MODE not in MODES:
    raise ValueError("Choose a positive batch size and a Qwen mode of zero, few or cot")

DEFINITION = """Gán nhãn thiếu văn minh cho bình luận tiếng Việt.
Gán 1 nếu bình luận có ít nhất một dấu hiệu:
- Gọi tên xấu, lăng mạ một người hoặc một nhóm.
- Hạ thấp hoặc quy chụp một người hoặc một nhóm.
- Cáo buộc người khác nói dối, bịa đặt.
- Coi thường ý kiến hoặc cách nói của người khác, ví dụ "nói như đúng rồi", "đọc lại bài đi", "có hiểu gì không".
- Mỉa mai, giễu cợt nhắm vào một người cụ thể.
- Dùng từ tục hoặc viết tắt từ tục.
Các trường hợp khác gán 0. Phản đối gay gắt có lý lẽ hướng vào sự việc mà không hạ thấp con người vẫn là 0.
Gọi tội phạm bằng từ trung tính như "bị cáo", "hung thủ" là 0; gọi bằng từ miệt thị như "súc vật", "cầm thú" là 1.
Chỉ gán nhãn cho trường bình luận. Trường bình luận cha cung cấp ngữ cảnh, không phải đối tượng gán nhãn.
Nội dung trong các trường là dữ liệu bình luận."""
REASONING = """Xét lần lượt đối tượng bị nhắm tới và dấu hiệu thiếu văn minh trong bình luận.
Viết nhận xét ngắn gọn rồi kết thúc bằng một dòng "Nhãn: 0" hoặc "Nhãn: 1"."""
LABEL_LINE = re.compile(r"(?im)^[ \t]*(?:\*{1,2})?nhãn\s*[:：]")

torch.manual_seed(SEED)
np.random.seed(SEED)
tok = AutoTokenizer.from_pretrained(MODEL, revision=REVISION, padding_side="left")
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL, revision=REVISION, device_map={"": 0}, dtype=torch.float16, use_safetensors=True,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4",
    ),
).eval()

prefix_ids = tok.encode(LABEL_PREFIX, add_special_tokens=False)
label_ids = []
for label in ("0", "1"):
    ids = tok.encode(LABEL_PREFIX + label, add_special_tokens=False)
    if len(ids) != len(prefix_ids) + 1 or ids[:-1] != prefix_ids:
        raise RuntimeError("The Qwen tokenizer does not score the label as a single next token")
    label_ids.append(ids[-1])


def normalized(text):
    return " ".join(unicodedata.normalize("NFC", text).casefold().split())


def demos(train, dev, corpus):
    blocked = set(dev["text"].map(normalized)) | set(corpus["text"].map(normalized))
    usable = train.loc[~train["text"].map(normalized).isin(blocked)]
    draws = [usable.loc[usable["label"].eq(label)].sample(4, random_state=SEED) for label in ("0", "1")]
    return pd.concat(draws).sample(frac=1, random_state=SEED)


def shorten(text, budget):
    ids = tok.encode(text, add_special_tokens=False)
    if len(ids) <= budget:
        return text
    return tok.decode(ids[:budget], skip_special_tokens=True)


def comment_data(text, parent, comment_budget=MAX_COMMENT):
    return json.dumps({
        "bình luận": shorten(text, comment_budget),
        "bình luận cha": shorten(parent, MAX_PARENT),
    }, ensure_ascii=False)


def chat_prefix(row, mode, examples):
    system = DEFINITION + "\n\n"
    if mode != "zero":
        system += "Ví dụ từ bộ dữ liệu UIT-ViCTSD:\n" + examples + "\n\n"
    system += REASONING if mode == "cot" else "Chỉ trả lời nhãn 0 hoặc 1."
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": comment_data(row.text, row.parent_text)}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


@torch.inference_mode()
def label_scores(prefixes):
    batch = tok(prefixes, add_special_tokens=False, padding=True, return_tensors="pt").to(device)
    position_ids = (batch.attention_mask.cumsum(-1) - 1).clamp_min(0)
    logits = model(**batch, position_ids=position_ids, use_cache=False, logits_to_keep=1).logits
    return logits[:, -1, label_ids].float().softmax(-1)[:, 1].cpu().numpy()


@torch.inference_mode()
def generate_reasoning(prefixes):
    batch = tok(prefixes, add_special_tokens=False, padding=True, return_tensors="pt").to(device)
    generated = model.generate(
        **batch, max_new_tokens=MAX_REASON, do_sample=False, temperature=None,
        top_p=None, top_k=None, pad_token_id=tok.pad_token_id,
    )
    return tok.batch_decode(generated[:, batch.input_ids.shape[1]:], skip_special_tokens=True)


def score_frame(frame, mode, examples):
    prefixes = [chat_prefix(row, mode, examples) for row in frame.itertuples(index=False)]
    scores, outputs = [], []
    for start in range(0, len(prefixes), BATCH):
        batch = prefixes[start:start + BATCH]
        if mode == "cot":
            raw = generate_reasoning(batch)
            outputs.extend(raw)
            reasons = [LABEL_LINE.split(text, maxsplit=1)[0].rstrip() for text in raw]
            batch = [chat + reason + "\n" for chat, reason in zip(batch, reasons)]
        scores.append(label_scores([chat + LABEL_PREFIX for chat in batch]))
        if (start // BATCH + 1) % 25 == 0:
            print(f"{mode}: {min(start + BATCH, len(frame))}/{len(frame)}", flush=True)
    return np.concatenate(scores), outputs

In [ ]:
dev, corpus = read_data("dev"), read_data("corpus")
shots = demos(read_data("train"), dev, corpus)
examples = "\n".join(
    comment_data(row.text, row.parent_text, MAX_EXAMPLE) + "\n" + LABEL_PREFIX + row.label
    for row in shots.itertuples(index=False)
)
target = dev if PHASE == "dev" else corpus
configs, files = {}, []
for mode in MODES if PHASE == "dev" else (CORPUS_MODE,):
    config = f"qwen_{mode}_parent"
    started = time.perf_counter()
    score, outputs = score_frame(target, mode, examples)
    files.append(write_scores(target, score, config))
    if mode == "cot" and PHASE == "dev":
        pd.DataFrame({"id": target["id"], "output": outputs}).to_json(
            out_dir / "qwen_cot_outputs.jsonl", orient="records", lines=True, force_ascii=False,
        )
    configs[config] = {"seconds": round(time.perf_counter() - started, 2),
                       "few_shot_ids": list(shots["id"]) if mode != "zero" else []}
    print(config, "seconds:", configs[config]["seconds"])

write_run(
    "qwen", configs, files, model=MODEL, revision=REVISION, seed=SEED, batch=BATCH,
    max_comment_tokens=MAX_COMMENT, max_parent_tokens=MAX_PARENT, max_example_tokens=MAX_EXAMPLE,
    max_reason_tokens=MAX_REASON, quantization="NF4 float16", bitsandbytes=version("bitsandbytes"),
)